# Lab 1 — Exploratory Modelling with the Lake Problem

## Background

The **Lake Problem** is a stylised decision problem in which the inhabitants of a lakeside town
decide how much phosphorus to release into a lake each year. If the pollution in the lake exceeds
a critical threshold, the lake undergoes irreversible eutrophication. The dynamics are governed by:

$$X_{t+1} = X_t + a_t + \frac{X_t^q}{1 + X_t^q} - bX_t + \varepsilon_t$$

where $X_t$ is the phosphorus concentration at time $t$, $a_t \in [0, 0.1]$ is the anthropogenic
release (the **decision lever**), $b$ is the natural removal rate, $q$ is the natural recycling
exponent, and $\varepsilon_t \sim \text{LogNormal}(\mu, \sigma)$ is natural inflow.

There are **four outcomes of interest**:

| Outcome | Formula | Direction |
|---------|---------|-----------|
| `max_P` | $\max_t X_t$ | minimise |
| `utility` | $\sum_t \alpha a_t \delta^t$ | maximise |
| `inertia` | fraction of years with $|a_t - a_{t-1}| > 0.2$ | minimise |
| `reliability` | fraction of years with $X_t < X_\text{crit}$ | maximise |

### What you will do in this lab

Using the **EMA Workbench**, you will:
1. Connect the lake model to the workbench by defining uncertainties, levers, and outcomes.
2. Perform open exploration under a *no-release* policy and visualise the outcomes.
3. Compare four randomly sampled release strategies.
4. Speed up computation using the `MultiprocessingEvaluator`.

### Deeply uncertain parameters

The model is subject to **deep uncertainty** about five parameters:

| Parameter | Range | Default | Meaning |
|-----------|-------|---------|---------|
| `mean` ($\mu$) | 0.01 – 0.05 | 0.02 | Mean of natural inflow distribution |
| `stdev` ($\sigma$) | 0.001 – 0.005 | 0.0017 | Std. dev. of natural inflow |
| `b` | 0.1 – 0.45 | 0.42 | Natural phosphorus removal rate |
| `q` | 2.0 – 4.5 | 2.0 | Natural recycling exponent |
| `delta` ($\delta$) | 0.93 – 0.99 | 0.98 | Economic discount rate |

## 1. Setup: imports and model definition

We start by importing the necessary libraries and connecting the lake model function to the
EMA Workbench. The `Model` object wraps the Python function; `RealParameter` objects define
both the uncertain factors and the decision levers; `ScalarOutcome` objects define the outputs.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from lakemodel_function import lake_problem
from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           Sample, perform_experiments, ema_logging)

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Instantiate the model ---
lake_model = Model('lakeproblem', function=lake_problem)
lake_model.time_horizon = 100   # 100 annual time steps

# --- Uncertain parameters (deep uncertainty) ---
lake_model.uncertainties = [
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('delta', 0.93,  0.99),
]

# --- Decision levers: one per time step (100 total) ---
# Each lever controls the anthropogenic release in year i, in [0, 0.1].
lake_model.levers = [RealParameter(f'l{i}', 0, 0.1)
                     for i in range(lake_model.time_horizon)]

# --- Outcomes ---
lake_model.outcomes = [
    ScalarOutcome('max_P'),
    ScalarOutcome('utility'),
    ScalarOutcome('inertia'),
    ScalarOutcome('reliability'),
]

## 2. Baseline exploration: no-release policy

We first explore the system behaviour with **no anthropogenic pollution** ($a_t = 0$ for all $t$).
This establishes a baseline: how does the lake behave under uncertainty alone, with no human
intervention?

We run **1 000 scenarios**, each with a different draw from the uncertain parameter ranges,
using the default Monte Carlo (random) sampler.

In [ ]:
# Fix all levers at 0 (no release policy)
no_release_policy = Sample('no release', **{l.name: 0 for l in lake_model.levers})

n_scenarios = 1000
results_baseline = perform_experiments(lake_model, n_scenarios, no_release_policy)

### 2.1 Pairplot of outcomes

A **pairplot** shows the scatter of every pair of outcomes across all scenarios. Diagonal panels
show the marginal distribution of each outcome. Look for:
- Trade-offs: negative correlations between outcomes (e.g., higher utility → lower reliability).
- Clusters or non-linearities that suggest threshold effects.

In [ ]:
experiments_b, outcomes_b = results_baseline

# Convert outcomes dict to DataFrame for plotting
outcomes_df = pd.DataFrame.from_dict(outcomes_b)
sns.pairplot(outcomes_df)
plt.suptitle('Outcomes under no-release policy (1 000 scenarios)', y=1.01)
plt.show()

### 2.2 Pairs scatter — outcomes coloured by uncertainty values

The EMA Workbench provides `pairs_scatter`, which adds colour coding by uncertainty value,
making it possible to visually identify which uncertain parameters drive system behaviour.

In [ ]:
from ema_workbench.analysis import pairs_plotting

pairs_plotting.pairs_scatter(experiments_b, outcomes_b)
fig = plt.gcf()
fig.set_size_inches(8, 8)
plt.show()

## 3. Multi-policy exploration

Now we sample **4 candidate strategies** from the lever space. Each strategy is a vector of
100 release decisions drawn at random. We run 1 000 scenarios for each, giving 4 000 total
model evaluations.

Compare the outcome distributions across policies: do different strategies shift the distribution?
Are there consistent trade-offs between `utility` and `reliability`?

In [ ]:
n_scenarios = 1000
n_policies = 4

# perform_experiments samples n_policies randomly from the lever space
results_multi = perform_experiments(lake_model, n_scenarios, n_policies)

In [ ]:
experiments_m, outcomes_m = results_multi

# Add policy column to outcomes for differentiated plotting
data = pd.DataFrame.from_dict(outcomes_m)
data['policy'] = experiments_m['policy']

# Pairplot with policy as hue
sns.pairplot(data, hue='policy', vars=list(outcomes_m.keys()))
plt.suptitle('Outcomes for 4 random policies (1 000 scenarios each)', y=1.01)
plt.show()

## 4. Parallelisation with MultiprocessingEvaluator

Running 4 000 model evaluations sequentially can take several minutes. The
`MultiprocessingEvaluator` distributes work across all available CPU cores.

> **Note for Windows users:** Python's `multiprocessing` requires the model function to be
> importable from a module (not defined inside the notebook). The `lakemodel_function.py` file
> satisfies this requirement, so multiprocessing works here.

In [ ]:
from ema_workbench import MultiprocessingEvaluator

n_scenarios = 1000
n_policies = 4

# The context manager handles pool creation and cleanup automatically.
with MultiprocessingEvaluator(lake_model) as evaluator:
    results_parallel = evaluator.perform_experiments(n_scenarios, n_policies)

experiments_p, outcomes_p = results_parallel
print(f"Experiments shape: {experiments_p.shape}")
print(f"Number of unique policies: {experiments_p['policy'].nunique()}")

## Summary

In this lab you:
- Connected a Python simulation model to the EMA Workbench.
- Ran open exploration experiments across a deep uncertainty space.
- Used pairplots and pairs-scatter to identify outcome trade-offs and sensitivity drivers.
- Compared baseline (no-release) behaviour against multiple candidate strategies.
- Used `MultiprocessingEvaluator` to speed up computation.

In **Lab 2** you will use global sensitivity analysis (Sobol and feature scoring) to
quantify *which* uncertain parameters are most influential for each outcome.